# Fine-tune Chronos-Bolt-Small on Vietnamese SJC Gold (Colab T4)

Phase 8 Option 3 of the TDTU NCKH 2025-2026 project — runs the per-fold fine-tune on a free Colab T4 GPU and downloads the trained checkpoints back to the user's machine.

**Inputs**: the public repo `https://github.com/twangnhat-05/NGHIENCUUKHOAHOC` (script + data already committed).
**Outputs**: `models/chronos_finetuned/fold_{0..4}/` HuggingFace directories + `chronos_finetuned_*.csv` benchmark tables.

**Expected wall-clock on T4**: 5 folds × 5 epochs × ~85 steps/epoch × ~0.1 s/step ≈ **5–8 min total**.

## Steps
1. Pick GPU runtime: `Runtime → Change runtime type → T4 GPU`.
2. Run cells in order.
3. Download `chronos_finetuned.zip` via the last cell.
4. Locally: `unzip chronos_finetuned.zip -d models/` and run `python -m scripts.benchmark_finetuned_chronos`.


## 1 — Verify GPU

In [ ]:
!nvidia-smi

## 2 — Clone repo and install deps

In [ ]:
!git clone https://github.com/twangnhat-05/NGHIENCUUKHOAHOC.git
%cd NGHIENCUUKHOAHOC
!git checkout wip/phase-8
!pip install -q chronos-forecasting==2.2.2 pyarrow scikit-learn lightgbm xgboost catboost statsforecast prophet shap

## 3 — Smoke test (2 min)

In [ ]:
!python -m scripts.finetune_chronos --fold 0 --device cuda --max-steps 30 --batch-size 16

## 4 — Full 5-fold fine-tune (5–8 min on T4)

In [ ]:
!rm -rf models/chronos_finetuned
!python -m scripts.finetune_chronos \
 --fold all \
 --device cuda \
 --epochs 5 \
 --batch-size 32 \
 --learning-rate 1e-5

## 5 — Benchmark fine-tuned vs zero-shot vs Ridge across all folds and horizons

In [ ]:
!python -m scripts.benchmark_finetuned_chronos --horizons 1 5 20

In [ ]:
import pandas as pd
summary = pd.read_csv('reports/leaderboard/chronos_finetuned_summary.csv')
print(summary.to_string(index=False))

## 6 — Zip the checkpoints + results and download to your machine

After the cell below completes, click the **folder** icon on the left, find `chronos_finetuned_artifacts.zip`, right-click `Download`.

In [ ]:
!zip -r chronos_finetuned_artifacts.zip \
 models/chronos_finetuned/ \
 reports/leaderboard/chronos_finetuned_long.csv \
 reports/leaderboard/chronos_finetuned_summary.csv \
 reports/leaderboard/chronos_finetune_log.csv
!ls -lh chronos_finetuned_artifacts.zip

## 7 — Local re-integration (run on your laptop)

```bash
cd D:\WangNhat\Study\NCKH
unzip ~/Downloads/chronos_finetuned_artifacts.zip
python -m scripts.benchmark_finetuned_chronos # local sanity check on CPU
```

Then re-render figures and update the paper:
```bash
python -m scripts.generate_paper_figures # picks up the new summary CSV
```